In [0]:
from pyspark.sql import functions as F

traffic = spark.table("workspace.default.silver_traffic")
weather = spark.table("workspace.default.silver_weather")
roads = spark.table("workspace.default.silver_road_edges")

In [0]:
gold_traffic = (
    traffic
    .groupBy("traffic_date", "traffic_hour")
    .agg(
        F.sum("probe_count").alias("total_probe_observations"),
        F.countDistinct("segment_id").alias("active_road_segments"),
        F.avg("probe_count").alias("average_probe_count"),
        F.avg("speed_limit").alias("average_speed_limit"),
        F.countDistinct("street_name").alias("unique_streets")
    )
)

In [0]:
gold_weather = (
    weather
    .groupBy("weather_date", "weather_hour")
    .agg(
        F.avg("temperature_2m").alias("avg_temperature"),
        F.avg("relative_humidity_2m").alias("avg_humidity"),
        F.sum("precipitation").alias("total_precipitation"),
        F.sum("rain").alias("total_rain"),
        F.avg("wind_speed_10m").alias("avg_wind_speed"),
        F.avg("cloud_cover").alias("avg_cloud_cover")
    )
)

In [0]:
gold_traffic_weather = (
    gold_traffic.alias("t")
    .join(
        gold_weather.alias("w"),
        (F.col("t.traffic_date") == F.col("w.weather_date")) &
        (F.col("t.traffic_hour") == F.col("w.weather_hour")),
        "left"
    )
    .select(
        F.col("t.traffic_date"),
        F.col("t.traffic_hour"),
        F.col("t.total_probe_observations"),
        F.col("t.active_road_segments"),
        F.col("t.average_probe_count"),
        F.col("t.average_speed_limit"),
        F.col("t.unique_streets"),
        F.col("w.avg_temperature"),
        F.col("w.avg_humidity"),
        F.col("w.total_precipitation"),
        F.col("w.total_rain"),
        F.col("w.avg_wind_speed"),
        F.col("w.avg_cloud_cover")
    )
)

In [0]:
(
    gold_traffic_weather.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_traffic_weather_hourly")
)

In [0]:
gold_road_network = (
    roads
    .groupBy("highway_type")
    .agg(
        F.count("*").alias("total_road_segments"),
        F.countDistinct("road_name").alias("unique_road_names"),
        F.avg("max_speed").alias("average_speed_limit"),
        F.sum(
            F.when(F.col("oneway") == "yes", 1).otherwise(0)
        ).alias("one_way_segments")
    )
)

In [0]:
(
    gold_road_network.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_road_network_summary")
)

In [0]:
crashes = spark.table("workspace.default.silver_crash_statistics")

gold_accidents = (
    crashes
    .select(
        F.col("year"),
        F.col("road_crashes"),
        F.col("road_crash_fatalities"),
        F.col("fatal_road_crashes")
    )
    .withColumn(
        "fatality_rate_per_100_crashes",
        F.when(
            F.col("road_crashes") > 0,
            F.col("road_crash_fatalities") * 100 / F.col("road_crashes")
        )
    )
)

In [0]:
(
    gold_accidents.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_accident_summary")
)

In [0]:
routes = spark.table("workspace.default.silver_routes")
trips = spark.table("workspace.default.silver_trips")
stops = spark.table("workspace.default.silver_stops")
stop_times = spark.table("workspace.default.silver_stop_times")

gold_transport = (
    routes.alias("r")
    .join(
        trips.alias("t"),
        F.col("r.route_id") == F.col("t.route_id"),
        "left"
    )
    .groupBy(
        F.col("r.route_id"),
        F.col("r.route_long_name"),
        F.col("r.route_type")
    )
    .agg(
        F.countDistinct("t.trip_id").alias("scheduled_trips")
    )
)

In [0]:
(
    gold_transport.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.gold_transport_summary")
)